In [1]:
!pip install groq -q
# 1. Instalar dependências (tudo gratuito!)
!pip install python-telegram-bot requests gtts nest_asyncio
!pip install playwright # opcional: automação de navegador

# 2. Instalar navegador para Playwright (opcional)
!python -m playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.18.0 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 17.3 MB/s eta 0:00:00
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 46.1s175.4 MiB [] 0% 22.8s175.4 MiB [] 0% 17.0s175.4 MiB [] 0

In [2]:
import os
import json
import requests
import nest_asyncio
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from groq import Groq
from google.colab import userdata

# ==========================================
# 1. RESOLVER CONFLITO DE ASSINCRONICIDADE NO COLAB
# ==========================================
nest_asyncio.apply()

# ==========================================
# 2. PUXAR CHAVES DOS SECRETS DO COLAB
# ==========================================
try:
    TELEGRAM_TOKEN = userdata.get('TELEGRAM_BOT_KEY')
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
except userdata.SecretNotFoundError as e:
    print(f"⚠️ AVISO: O secret não foi encontrado. Verifique se você cadastrou e ativou o acesso no menu lateral.")
    raise e

# Inicializa o cliente Groq com a chave segura
client = Groq(api_key=GROQ_API_KEY)

# ==========================================
# 3. DEFINIÇÃO DAS FUNÇÕES (TOOLS PYTHON)
# ==========================================

def consultar_clima(cidade: str) -> str:
    """Consulta o clima atual usando wttr.in"""
    try:
        response = requests.get(f"https://wttr.in/{cidade}?format=j1")
        data = response.json()
        atual = data['current_condition'][0]
        return f"Clima em {cidade}: {atual['temp_C']}°C, Condição: {atual['weatherDesc'][0]['value']}, Sensação térmica de {atual['FeelsLikeC']}°C."
    except Exception as e:
        return f"Erro ao buscar clima para {cidade}. Tente novamente mais tarde."

def buscar_noticias(tema: str) -> str:
    """Busca as 3 principais notícias sobre um tema no NewsAPI"""
    try:
        url = f"https://newsapi.org/v2/everything?q={tema}&language=pt&sortBy=publishedAt&pageSize=3&apiKey={NEWS_API_KEY}"
        response = requests.get(url)
        data = response.json()

        if data.get("status") != "ok" or not data.get("articles"):
            return f"Nenhuma notícia recente encontrada sobre o tema: {tema}."

        artigos = [f"- {a['title']}: {a['description']}" for a in data["articles"]]
        return "\n".join(artigos)
    except Exception as e:
        return f"Erro ao buscar notícias sobre {tema}."

def resumir(texto: str) -> str:
    """Usa o Groq para resumir um texto passado"""
    try:
        resposta = client.chat.completions.create(
            model="llama-3.3-70b-versatile", # Atualizado para o novo modelo
            messages=[
                {"role": "system", "content": "Você é um especialista em resumos. Resuma o texto fornecido pelo usuário de forma clara, direta e em português."},
                {"role": "user", "content": texto}
            ]
        )
        return resposta.choices[0].message.content
    except Exception as e:
        return "Erro ao tentar resumir o texto."

available_functions = {
    "consultar_clima": consultar_clima,
    "buscar_noticias": buscar_noticias,
    "resumir": resumir,
}

# ==========================================
# 4. CONFIGURAÇÃO DAS TOOLS NO GROQ
# ==========================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "consultar_clima",
            "description": "Obtém a previsão do tempo atual para uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cidade": {"type": "string", "description": "O nome da cidade, ex: São Paulo, Rio de Janeiro"}
                },
                "required": ["cidade"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_noticias",
            "description": "Busca as últimas manchetes e notícias sobre um tema ou assunto específico.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tema": {"type": "string", "description": "O tema da notícia, ex: tecnologia, economia, inteligência artificial"}
                },
                "required": ["tema"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "resumir",
            "description": "Resume um texto longo fornecido pelo usuário.",
            "parameters": {
                "type": "object",
                "properties": {
                    "texto": {"type": "string", "description": "O texto ou bloco de texto que precisa ser resumido."}
                },
                "required": ["texto"],
            },
        },
    }
]

# ==========================================
# 5. LÓGICA DO BOT (LLM + STT)
# ==========================================

async def process_user_input(user_message: str, update: Update):
    """Função central que processa tanto o texto digitado quanto o transcrito do áudio"""
    messages = [
        {"role": "system", "content": "Você é um assistente de Telegram útil e amigável. Use as ferramentas disponíveis para responder sobre o clima, buscar notícias ou resumir textos. Entregue a resposta final em português de forma natural e bem formatada."},
        {"role": "user", "content": user_message}
    ]

    try:
        # Atualizado para o modelo llama-3.3-70b-versatile
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            messages.append(response_message)

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_functions[function_name]

                try:
                    function_args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
                except json.JSONDecodeError:
                    function_args = {}

                function_response = function_to_call(**function_args)

                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                })

            # Segunda chamada com os resultados das tools (também atualizada para 3.3)
            second_response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages
            )
            final_answer = second_response.choices[0].message.content
        else:
            final_answer = response_message.content

        await update.message.reply_text(final_answer)

    except Exception as e:
        print(f"Erro na API Groq: {e}")
        await update.message.reply_text("Desculpe, meus circuitos falharam por um momento. Pode tentar novamente?")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Recebe mensagens de texto e as envia para o processamento"""
    await process_user_input(update.message.text, update)

async def handle_audio(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Recebe mensagens de voz, transcreve com Whisper e envia para o processamento"""
    status_msg = await update.message.reply_text("🎧 Ouvindo e transcrevendo seu áudio...")

    try:
        # Pega a referência do arquivo no Telegram
        audio_file = update.message.voice or update.message.audio
        file = await context.bot.get_file(audio_file.file_id)

        # Cria um caminho temporário no Colab
        file_path = f"temp_audio_{audio_file.file_id}.ogg"
        await file.download_to_drive(file_path)

        # Envia o arquivo para a API do Groq (Whisper)
        with open(file_path, "rb") as f:
            transcription = client.audio.transcriptions.create(
              file=(file_path, f.read()),
              model="whisper-large-v3",
              language="pt" # Força o idioma português para maior precisão
            )

        user_text = transcription.text

        # Atualiza o status para mostrar o que o bot "ouviu"
        await status_msg.edit_text(f"🗣️ *Você disse:* _{user_text}_\n\nProcessando...", parse_mode='Markdown')

        # Remove o áudio temporário para não lotar o disco do Colab
        os.remove(file_path)

        # Manda o texto transcrito para o cérebro principal
        await process_user_input(user_text, update)

        # Limpa a mensagem de status após enviar a resposta final
        await status_msg.delete()

    except Exception as e:
        print(f"Erro no processamento de áudio: {e}")
        await status_msg.edit_text("Desculpe, tive um problema ao tentar entender o seu áudio.")

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    boas_vindas = (
        "👋 Olá! Eu sou seu Bot Assistente.\n\n"
        "Você pode me enviar **textos ou áudios** pedindo coisas como:\n"
        "🌤️ *'Qual o clima em Curitiba?'*\n"
        "📰 *'Busque notícias sobre Inteligência Artificial'* ou\n"
        "📝 *'Resuma este texto: [seu texto aqui]'*\n\n"
        "Mande um áudio e vamos testar!"
    )
    await update.message.reply_text(boas_vindas, parse_mode='Markdown')

# ==========================================
# 6. INICIALIZAÇÃO NO COLAB
# ==========================================

app = Application.builder().token(TELEGRAM_TOKEN).build()

app.add_handler(CommandHandler("start", start))
# Handler de Texto
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
# Handler de Áudio e Voz
app.add_handler(MessageHandler(filters.VOICE | filters.AUDIO, handle_audio))

print("🤖 Bot iniciado com Llama-3.3-70b e Whisper STT! (Pare a célula para desligar)")

app.run_polling(allowed_updates=Update.ALL_TYPES, close_loop=False)

🤖 Bot iniciado com Llama-3.3-70b e Whisper STT! (Pare a célula para desligar)
